# Binary predictors: observed false versus unknown

This focused notebook completes the initial **binary** review of
`public_meeting`, `permit`. It describes the supplied training and test predictors,
then uses the labelled training rows to identify relationships worth
carrying into a leakage-safe modelling pipeline.

Missing values remain a third auditable state; they are not silently coerced to `False`.

This is exploratory evidence, not fitted preprocessing. Category pooling,
imputation, encoding and scaling must be learned inside each training fold.


## Consistent audit contract

Every focused predictor audit answers the same questions before adding
type-specific checks:

1. What is explicitly missing, and what looks like a sentinel?
2. What range or category coverage is present in training and test?
3. How much of the test set is exposed to unseen training levels?
4. Does the labelled distribution vary enough to justify retaining the field?
5. What exact baseline treatment follows from the evidence?

Target-rate tables flag support rather than treating tiny groups as reliable.
Train/test comparisons are descriptive and do not use the hidden test labels.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

source_directory = str(Path("../src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    MISSING_CATEGORY,
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    cramer_v,
    hierarchy_conflicts,
    hierarchy_summary,
    numeric_summary,
    numeric_target_summary,
    normalise_categories,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = Path("../data")
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)
audited_features = ['public_meeting', 'permit']
assert set(audited_features).issubset(training_features.columns)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {len(audited_features)} predictors."
)


Validated 59,400 training rows and 14,850 test rows for 2 predictors.


## 1. Value coverage and missingness


In [2]:
display(categorical_summary(training_features, test_features, audited_features))
for feature in audited_features:
    print()
    print(feature)
    display(category_frequency_table(training_features, test_features, feature, top_n=5))


,training explicit missing,training source blank rows,training sentinel rows,test explicit missing,test source blank rows,test sentinel rows,training levels,test levels,training levels with <20 rows,training rows in rare levels (%),test-only levels,test rows in unseen levels (%),training-only levels,marginal total-variation distance
feature,,,,,,,,,,,,,,
public_meeting,0,3334,0,0,821,0,2,2,0,0.0,0,0.0,0,0.0018
permit,0,3056,0,0,737,0,2,2,0,0.0,0,0.0,0,0.0028



public_meeting


,training rows,training (%),test rows,test (%)
public_meeting,,,,
true,51011,85.88,12738,85.78
false,5055,8.51,1291,8.69
<missing/blank>,3334,5.61,821,5.53



permit


,training rows,training (%),test rows,test (%)
permit,,,,
true,38852,65.41,9754,65.68
false,17492,29.45,4359,29.35
<missing/blank>,3056,5.14,737,4.96


## 2. Relationship with `status_group`

The class percentages are conditional on each true/false/unknown state.
Row support is shown before any interpretation of percentage differences.


In [3]:
for feature in audited_features:
    print()
    print(feature)
    display(
        categorical_target_profile(
            training_data,
            feature,
            minimum_support=100,
        )
    )



public_meeting


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
public_meeting,,,,,
true,51011,True,55.69,7.29,37.02
false,5055,True,42.99,8.74,48.27
<missing/blank>,3334,True,50.33,4.68,44.99



permit


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
permit,,,,,
true,38852,True,55.44,6.94,37.61
false,17492,True,51.71,7.55,40.74
<missing/blank>,3056,True,54.74,9.82,35.44


## 3. Training/test stability


In [4]:
binary_stability = categorical_summary(
    training_features,
    test_features,
    audited_features,
)[[
    "training explicit missing",
    "test explicit missing",
    "test-only levels",
    "test rows in unseen levels (%)",
    "marginal total-variation distance",
]]
display(binary_stability)


,training explicit missing,test explicit missing,test-only levels,test rows in unseen levels (%),marginal total-variation distance
feature,,,,,
public_meeting,0,0,0,0.0,0.0018
permit,0,0,0,0.0,0.0028


## Decision register

The register separates observed evidence from the proposed baseline action.
A retained field is still a candidate: later validation must show whether it
improves generalisation and whether a coarser related representation is safer.


In [5]:
decision_register = pd.DataFrame([{'feature': 'public_meeting', 'quality finding': 'The source is blank for 3,334 training rows (5.61%) and 821 test rows.', 'baseline treatment': 'Retain true/false/unknown as three explicit states.', 'risk to verify': 'Missingness may reflect the data-collection process rather than pump condition.'}, {'feature': 'permit', 'quality finding': 'The source is blank for 3,056 training rows (5.15%) and 737 test rows.', 'baseline treatment': 'Retain true/false/unknown as three explicit states.', 'risk to verify': 'Permit status may proxy geography or administration.'}])
display(decision_register.set_index("feature"))


,quality finding,baseline treatment,risk to verify
feature,,,
public_meeting,"The source is blank for 3,334 training rows (5.61%) and 821 test rows.",Retain true/false/unknown as three explicit states.,Missingness may reflect the data-collection process rather than pump condition.
permit,"The source is blank for 3,056 training rows (5.15%) and 737 test rows.",Retain true/false/unknown as three explicit states.,Permit status may proxy geography or administration.


### Handoff to modelling

Use nullable booleans or an explicit unknown category; never apply Python truthiness to missing values.

- Preserve raw source frames and implement the stated sentinel rules on copies.
- Fit imputers, rare-level grouping and encoders on each training fold only.
- Map unseen validation or test categories to an explicit fallback.
- Compare the stated baseline treatment with a simple omission ablation.
- Revisit target-rate observations after the reproducible stratified split exists.
